<a href="https://colab.research.google.com/github/gaeun1961/DS_Basic_Analysis_Python/blob/main/Ch05_01_%EB%84%A4%EC%9D%B4%EB%B2%84_api%EB%A5%BC_%EC%9D%B4%EC%9A%A9%ED%95%9C_%ED%81%AC%EB%A1%A4%EB%A7%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import urllib.request
import datetime
import time
import json
import urllib.parse

In [2]:
from google.colab import userdata
client_id = userdata.get('naver_client_id')
client_secret = userdata.get('naver_client_secret')

**1. API 요청 및 응답 수신**

네이버 API 서버에 접속하기 위해 인증 헤더를 설정하고, 서버로부터 응답 데이터를 받아오는 가장 기초적인 함수

In [3]:
## [CODE 1]
def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)

    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')

    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

**2. 검색 URL 생성 및 호출**

사용자가 입력한 검색어와 조건(시작 위치, 노출 개수)을 조합하여 API 호출 규격에 맞는 URL을 생성하고 결과를 JSON으로 변환

In [4]:
## [CODE 2]
def getNaverSearch(node, srcText, start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(srcText), start, display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)  # [CODE 1] 함수를 통해 데이터 수신

    if (responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)

**3. 필요한 데이터 항목 추출**

API 응답 결과에는 불필요한 정보도 포함되어 있기에 제목, 설명, 링크, 날짜 등 분석에 꼭 필요한 정보만 골라내어 정리

In [5]:
## [CODE 3]
def getPostData(post, jsonResult, cnt):
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    pData = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900')
    pDate = pData.strftime('%Y-%m-%d %H:%M:%S')

    jsonResult.append({'cnt':cnt, 'title':title, 'description':description, 'org_link':org_link, 'link':link, 'pDate':pDate})

**4. 크롤링 전체 프로세스 제어 및 저장**

사용자로부터 검색어를 입력받고, 네이버 뉴스 정책(최대 1,000건)에 맞춰 반복 수집을 진행한 뒤 최종 결과를 JSON 파일로 저장

In [6]:
## [CODE 0]
def main():
    node = 'news'  # 검색 카테고리 설정
    srcText = input('검색어를 입력하세요: ')
    cnt = 0
    jsonResult = []

    jsonResponse = getNaverSearch(node, srcText, 1, 100)
    total = jsonResponse['total']

    while ((jsonResponse != None) and (jsonResponse['display'] != 0)):
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt)

        start = jsonResponse['start'] + jsonResponse['display']
        if start > 1000: break  # 네이버는 1000개까지만 조회를 허용함
        jsonResponse = getNaverSearch(node, srcText, start, 100)

    print('전체 검색: %d 건' % total)

    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding='utf8') as outfile:
        jsonFile = json.dumps(jsonResult, indent=4, sort_keys=True, ensure_ascii=False)
        outfile.write(jsonFile)

    print("가져온 데이터 : %d 건" % (cnt))
    print('%s_naver_%s.json SAVED' % (srcText, node))

if __name__ == '__main__':
    main()

검색어를 입력하세요: 헬스케어
[2026-01-20 07:53:38.704743] Url Request Success
[2026-01-20 07:53:40.071806] Url Request Success
[2026-01-20 07:53:41.421538] Url Request Success
[2026-01-20 07:53:42.789883] Url Request Success
[2026-01-20 07:53:44.163376] Url Request Success
[2026-01-20 07:53:45.507630] Url Request Success
[2026-01-20 07:53:46.863169] Url Request Success
[2026-01-20 07:53:48.226311] Url Request Success
[2026-01-20 07:53:49.596207] Url Request Success
[2026-01-20 07:53:50.966642] Url Request Success
전체 검색: 1251463 건
가져온 데이터 : 1000 건
헬스케어_naver_news.json SAVED
